In [ ]:
# -*- coding: utf-8 -*-

In [ ]:
"""ML Flow and Supervised Learning.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1bfYdKVNFVPO-mFvrLvHq3I9qIDzkPV5o

**Business Understanding**

**Problem Statement**: Predict which telecom customers are at high risk of churning.

**Business Objective**: Reduce churn, increase retention, and increase customer lifetime value (CLV).

**Success Metrics**: Recall, ROC-AUC, Revenue Saved.
Target Variable: TARGET = "Churn"

In [ ]:
"""

#Environment Setup
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import joblib

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler, RobustScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, roc_curve

warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid")

#Data Loading
df = pd.read_csv('dataset.csv')
display(df.head())

#Audit Report
audit_report = pd.DataFrame({
    "dtype": df.dtypes,
    "missing": df.isnull().sum(),
    "missing_pct": round(df.isnull().mean()*100,2),
    "unique": df.nunique(),
    "duplicates": df.duplicated().sum()
})

display(audit_report)

#Data Dictionary
data_dictionary = {
    "State": "Customer's state/location",
    "Account length": "Number of days customer has been associated with the telecom company",
    "Area code": "Customer's telecom area code",
    "International plan": "Whether customer has subscribed to international calling plan (Yes/No)",
    "Voice mail plan": "Whether customer has subscribed to voicemail service (Yes/No)",
    "Number vmail messages": "Total number of voicemail messages",
    "Total day minutes": "Total daytime call minutes used by customer",
    "Total day calls": "Total number of daytime calls",
    "Total day charge": "Charges incurred during daytime usage",
    "Total eve minutes": "Total evening call minutes used by customer",
    "Total eve calls": "Total number of evening calls",
    "Total eve charge": "Charges incurred during evening usage",
    "Total night minutes": "Total nighttime call minutes used by customer",
    "Total night calls": "Total number of nighttime calls",
    "Total night charge": "Charges incurred during nighttime usage",
    "Total intl minutes": "Total international call minutes used",
    "Total intl calls": "Total number of international calls",
    "Total intl charge": "Charges incurred from international calls",
    "Customer service calls": "Number of calls made to customer support",
    "Churn": "Target variable indicating whether customer left the company (True/False)"
}

data_dict_df = pd.DataFrame(
    list(data_dictionary.items()),
    columns=["Column Name", "Business Meaning"]
)

display(data_dict_df)

#Missing Value Handling
# Mean (balanced data) - Median(inbalanced data) - Drop
#Duplicate Handling
#drop_duplicates

#Business KPI Layer
df['Total Charges'] = df['Total day charge'] + df['Total eve charge'] + df['Total night charge'] + df['Total intl charge']
avg_revenue_per_customer = df['Total Charges'].mean()
clv = avg_revenue_per_customer * 12
print(f"Churn Rate: {df['Churn'].mean() * 100:.2f}%")
print(f"Estimated Annual CLV: ₹{clv:.2f}")

kpis = pd.DataFrame({
    "Metric":[
        "Customers",
        "Churn %",
        "Avg Revenue",
        "Annual CLV"
    ],
    "Value":[
        len(df),
        round(df["Churn"].mean()*100,2),
        round(avg_revenue_per_customer,2),
        round(clv,2)
    ]
})
print(kpis)

#Univariate, Categorical & Skewness Analysis

import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Univariate: Distribution, KDE, Boxplot
fig, ax = plt.subplots(1, 3, figsize=(18, 5))
sns.histplot(df['Total day minutes'], kde=True, ax=ax[0]).set_title('Day Minutes Dist')
sns.kdeplot(df['Total Charges'], fill=True, ax=ax[1]).set_title('Total Charges KDE')
sns.boxplot(x=df['Customer service calls'], ax=ax[2]).set_title('Service Calls Boxplot')
plt.show()

# Categorical
sns.countplot(x='International plan', hue='Churn', data=df)
plt.show()

# Skewness
print(df.select_dtypes(include='number').skew().sort_values(ascending=False))

#We need to identify exactly which raw features drive churn before building any models

#Standard Bivariate
fig, axes = plt.subplots(figsize=(8, 6)) # Create a single figure and axes
sns.violinplot(x='Churn', y='Total Charges', data=df, ax=axes)
axes.set_title("Violin Plot: Total Charges by Churn")
plt.show()

# Explicit top correlations
print("Top Features Correlated with Churn")

# Ensure 'Churn' column is numerical (0 or 1)
df['Churn'] = df['Churn'].astype(int)

# Calculate correlation matrix for all numerical columns in df, including 'Churn'
correlation_with_churn = df.corr(numeric_only=True)['Churn'].sort_values(ascending=False).head(5)
print(correlation_with_churn)

# Churn by State BEFORE encoding
state_churn = df.groupby('State')['Churn'].mean().sort_values(ascending=False)
display(state_churn.head().round(3))

plt.figure(figsize=(12, 6))
sns.barplot(x=state_churn.head(10).index, y=state_churn.head(10).values, palette='viridis')
plt.title('Top 10 States by Churn Rate')
plt.xlabel('State')
plt.ylabel('Churn Rate')
plt.xticks(rotation=45, ha='right')
plt.show()

#Churn Rate by Service Calls
churn_by_calls = (df.groupby("Customer service calls")["Churn"].mean())
fig, ax = plt.subplots(figsize=(10, 6))
churn_by_calls.plot(kind="bar", color='coral', ax=ax)
ax.set_title("Churn Rate by Service Calls")
ax.set_ylabel("Churn Rate")
plt.show()

#Multivariate Analysis

#Pairplot
sns.pairplot(
    df[["Total day minutes", "Total Charges", "Customer service calls", "Churn"]],
    hue="Churn",
    palette="Set2"
)
plt.show()

#Correlation Heatmap & Crosstab
plt.figure(figsize=(10,6))
sns.heatmap(df.corr(numeric_only=True), annot=False, cmap='coolwarm')
plt.show()
display(pd.crosstab(df['International plan'], df['Churn'], normalize='index').round(2))

#Outlier Handling for All Numerical Columns

numeric_cols = df.select_dtypes(include=np.number).columns

print("Applying IQR-based clipping to numerical columns:")
for col in numeric_cols:
    if col == 'Churn': # Skip the target variable if it's numerical (boolean converted to int)
        continue
    Q1, Q3 = df[col].quantile([0.25, 0.75])
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    initial_min = df[col].min()
    initial_max = df[col].max()
    df[col] = df[col].clip(lower=lower_bound, upper=upper_bound)
    final_min = df[col].min()
    final_max = df[col].max()

    if initial_min != final_min or initial_max != final_max:
        print(f"  - Column '{col}': Outliers clipped. Min changed from {initial_min:.2f} to {final_min:.2f}, Max changed from {initial_max:.2f} to {final_max:.2f}")
    else:
        print(f"  - Column '{col}': No outliers found or already within bounds.")

#We cap (winsorize) outliers instead of deleting them to avoid losing valuable customer data.
#Visualizing the transformation proves it worked

#Algorithms learn better from explicit signals. We use pd.qcut to create smart business segments

#Feature Engineering
df['Total_Usage'] = df['Total day minutes'] + df['Total eve minutes'] + df['Total night minutes'] + df['Total intl minutes']
df['Service_Stress'] = df['Customer service calls'] / (df['Account length'] + 1)

# Using pd.qcut as requested (Expert Review Addition)
df['Revenue_Segment'] = pd.qcut(df['Total Charges'], q=3, labels=['Low', 'Medium', 'High'])

print(df.columns)

#Machine learning models only read numbers.
#We must track our matrix shape to ensure one-hot encoding doesn't cause a dimensional explosion

le = LabelEncoder()
df['International plan'] = le.fit_transform(df['International plan'])
df['Voice mail plan'] = le.fit_transform(df['Voice mail plan'])
df['Churn'] = le.fit_transform(df['Churn'])
df = pd.get_dummies(df, columns=['State', 'Revenue_Segment'], drop_first=True, dtype=int)

#Feature Selection

# Automated High Correlation Detection
corr_matrix = df.corr(numeric_only=True).abs()
high_corr = set()
for i in range(len(corr_matrix.columns)):
    for j in range(i):
        if corr_matrix.iloc[i, j] > 0.95:
            high_corr.add(corr_matrix.columns[i])

print(f"Dropping highly correlated data-driven columns: {high_corr}")
df.drop(columns=high_corr, inplace=True)

area_churn = (
    df.groupby("Area code")["Churn"]
      .mean()
)

print(area_churn)

# Drop Area Code
df.drop('Area code', axis=1, inplace=True, errors='ignore')
print("Note: 'Area code' was dropped because it acts as an arbitrary ID and has no predictive power for churn.")

#Train-Test Split & Scaling Comparison

X = df.drop(['Churn'], axis=1)
y = df['Churn']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# Compare Scalers
scaler_std = StandardScaler()
scaler_minmax = MinMaxScaler()
scaler_robust = RobustScaler()

X_train_std = pd.DataFrame(scaler_std.fit_transform(X_train), columns=X_train.columns)
X_test_std = pd.DataFrame(scaler_std.transform(X_test), columns=X_test.columns)

X_train_mm = pd.DataFrame(scaler_minmax.fit_transform(X_train), columns=X_train.columns)
X_train_rob = pd.DataFrame(scaler_robust.fit_transform(X_train), columns=X_train.columns)

print("StandardScaler Mean:", X_train_std['Total day minutes'].mean().round(2))
print("MinMaxScaler Min-Max:", X_train_mm['Total day minutes'].min(), "-", X_train_mm['Total day minutes'].max())

#Class Distribution Visualization
#If we don't know the class imbalance, we won't understand why our baseline Accuracy is misleading

# Visualizing Imbalance
plt.figure(figsize=(5,5))
y_train.value_counts().plot(kind='pie', autopct='%1.1f%%', colors=['#66b3ff','#ff9999'])
plt.title("Class Distribution (Churn vs Retained)")
plt.ylabel("")
plt.show()

# Feature Distribution Before vs After Scaling
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(X_train["Total day minutes"], ax=axes[0], kde=True, color='gray').set_title("Before Scaling")
sns.histplot(X_train_std["Total day minutes"], ax=axes[1], kde=True, color='green').set_title("After Scaling")
plt.show()

#Models, Evaluation & Confusion Matrix

# Train Logistic & Decision Tree
lr = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
lr.fit(X_train_std, y_train)
pred_lr, prob_lr = lr.predict(X_test_std), lr.predict_proba(X_test_std)[:, 1]
print(f"Logistic Regression -> Recall: {recall_score(y_test, pred_lr):.3f}")

dt = DecisionTreeClassifier(max_depth=5, random_state=42, class_weight='balanced')
dt.fit(X_train_std, y_train)
pred_dt, prob_dt = dt.predict(X_test_std), dt.predict_proba(X_test_std)[:, 1]
print(f"Decision Tree -> Recall: {recall_score(y_test, pred_dt):.3f}")

# Train Random Forest
rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    class_weight='balanced'
)

rf.fit(X_train_std, y_train)

pred_rf = rf.predict(X_test_std)
prob_rf = rf.predict_proba(X_test_std)[:, 1]

print(f"Random Forest -> Recall: {recall_score(y_test, pred_rf):.3f}")

# Confusion Matrix Heatmap
fig, axes = plt.subplots(1, 3, figsize=(20, 5))

# Logistic Regression
sns.heatmap(
    confusion_matrix(y_test, pred_lr),
    annot=True,
    fmt='d',
    cmap='Blues',
    ax=axes[0]
)
axes[0].set_title("Logistic Regression")

# Decision Tree
sns.heatmap(
    confusion_matrix(y_test, pred_dt),
    annot=True,
    fmt='d',
    cmap='Oranges',
    ax=axes[1]
)
axes[1].set_title("Decision Tree")

# Random Forest
sns.heatmap(
    confusion_matrix(y_test, pred_rf),
    annot=True,
    fmt='d',
    cmap='Greens',
    ax=axes[2]
)
axes[2].set_title("Random Forest")

plt.tight_layout()
plt.show()

In [ ]:
"""**ACCURACY**

100 CUSTOMERS - MODEL PREDICTION
![image.png](data:image/png;base64,iVBORw0KGgoAAAANSUhEUgAAA58AAAETCAYAAACx/LL2AAAAAXNSR0IArs4c6QAAAARnQU1BAACxjwv8YQUAAAAJcEhZcwAAEnQAABJ0Ad5mH3gAAB6VSURBVHhe7d1RaFz3nS/wny55UCCBEfTBgi5EJSwek0Btsg8x5CEjcmEd8tCYXahNC7a0hb1OLqjjXLh1modcZy+0lg1dZx96RzI02Ast9kKDXWiY8UPBeWixH1xmDA2WIYUxJDADCUiQwP8+nBlp5mjkWEpOsaTPB34gnXNmRin41/93zv//P2MRkQIAAAAK9N/yBwAAAOCbJnwCAABQOOETAACAwgmfAAAAFE74BAAAoHDCJwAAAIUTPgEAACic8AkAAEDhhE8AAAAKJ3wCAABQOOETAACAwgmfAAAAFE74BAAAoHDCJwAAAIUTPgGIWGxGSilSq5Y/A7A9zNWjk1KkTj2q/WPPnYjzvzgR+4ev3Jpv8r1gl9py+Nz/ei3qrU4sf5GyAUtKkZY70f7DmajkLwbYlWbiarvXHz+7EafzpwF2lFo0+2PCgVrutOPmr07EVP7ywh2Jq++fjxOvn4/G70/kT27SN/lesHttIXzuj1PX2nHjFzNR2VuK8cciYmUlVlYiYrwUe57Z/w1/I1SJ6q9vxN1PlqO5mD8H8AibOxIH9/R+fuL5OHQ2d/4hzfyiHs2PO9H+/ep3+QCPtpWVWPk8Gx+Ol/bE/h+cj5uN6t84gN6P9qcrEbES7Xut/MmNHTsf9dvt6LQH7qBu9b2AIZsOn9XfN+L0P+6J8ViJpevvxuw/jMXY44/H44+PxXeOzEfjTjdW8i/6WvbHoZeej6lvjedPADzSTr16MEoRcf/+/YiI2P/i+S0NvJ5/qRLlb/e+7AN45HWj8ZPH4/Ens/HhyxdasRIRpRdPxOmX8tcWqRGzzz4eY2OPx74fNfInN/ZCJSrP7InS0NBzi+8FDNlc+HzpYpx4qRQRK9G6cDi+U3ktFv60dnrpP0/G9POH493B1wDsSqfi5e+OR8RS3PjJjViKiHjmYFSfzl8HsLNdO74QN7oREVNR/sf8WWA32VT4PPJ6JfvW/qMr8drxa/nTQ6qNzsjNK/rHO43+RIb9ceJXN6PdGVgf0GnGxR/0N8A4E5VSdmX5WHZ+9bVPn4jzf7gbneXBdQV3o372yMDdhWrUOylS6kT9jRNx8Xbv70opOrcvxomnI6beuBzNT3rv8cVy3G2ciUOrr88+5+If29Hpr2/9Yjl77XO984ML3P/1cjQ/S5FSM2zbAbvY2y/H/ici4t6tePfCu3HrXkQ8tj8qPxlx7/PpI3HmWnOol6Xlu3H5YtZbZvZml5VePDPQV/trqzpRnxt8s/7xtR409f0zcfV2e7hXftKMi/864m8BKNB4KdaNzS63liOltLq8aupfL8bNj9fGa2m5E81fDW/0M/X9M1FfWrtm+ZObcX5y4IKeWis7P7x0qz/2zD63/xk3ftb7u46Vs8tKlTgz0E9Hv9ehOHUl37/X/71rr90fp67dXRtTdu7G1Tf0YnaX9LBVa6WUUkp3f11Zdy5f1UYnu7hVG3m806imiEgzv21n133RSXdvN1Pz9t3U+aKT6nOR4qeXe79nlyx/3EzN281045dHUjxdTfVPsuOrr/14uXcgpfaVmd5nVlO9k1JKy6nzyXJKn9xNzaXe35ZSWr51M91NKXWWmqnZXnt9c7H33zjwOZ2l7PNXr2tfTTMRKebqqZNSSp1maq6+dTPVRvzvopTaDTWVzt/KOkG/X1Z+fTc78JeLqTJ47dPVVO+1wbTaZ+6mznI71S/W0o3bzdT+rHfyk7tZD7pyKkXUUjN7RdYvV9+zf3ytB1UbnZS+WE7tVq6HLd9MZ57uvW4xe1W+Zyul1MPXBn3p6fPp5he946/HwNisk5qttTFZczHS1Bu9MdXquLCZVoddv+2N7V6qpWZ/yNa77u4nKaXl3sFOPVV7n90fuzYX+3/PoVS7vTbe648t253l1Fw8kmp/GBhPrv4Nl9Opjd6r1b+212MHx5h/PJOmcn9H55NOSsvtof+uoV6s1M6vdQc2rPX/6Dauhw2fq4H2V1Nr1z13JB15qf+afoMa/txTf+j9i/2knqoD/2APLTbTckopfXEznX96+PXpLxfTof51/YFg72/JmsPagDHdzv7u/t9791eHBv47ZtLVdkopLacbbw+Ez5TS8l8upxPPrf09SqldWKsDrbvpcr+XvXQ5ZV3nbrq42t8GvoD77GY6/8qI9xoctPT6ZlYbDPJGhM8jb5/J9aW1vnjzbO+Y8KmU+tq1vi/tf+VUuvyX3Jf2g2Oz5bvp8uv7e6/vHf/ibro42A+PXU3tlFL67EY6PTgG/PhqOrE6BtyfTv2h96YPCJ9TZ2/2xontdPWNgbHnYPX74cD7jHqvyqXeWHK5mWoDf+9qgE7tdPXY8GtTZ2Dcuvr/FQO9WKkdXpuadts3/tg3Nz1g6dNse6Kpf7oRN6+cjpnnIuJPl+LSB/krB52Iyt5sFfit96Zj/qO1M6vrCh7bHwf/x9rxiIhb7x+N/mThax8uRTciIpai8X/ns/VYsRTzt7Of4rGIiBNx6Nlszu/UD66uTadItTi0JyJiPErf7r1hRETcj8a/HY53B9bBArvP1E8qsf+x3pTbfi/7oDf1Nqbi+X/pP5DqSLz6D9l2uLcuHI7X3u+/wzfr0lsXI354NW7cbka7sxzLX6wtZ8imwAF8k0pROZuNmW7+9nS8+vR4xEorFn70cizkrrx//Z04/O+3sl9ePxQHShHx2FQc+e3ANNbFQ7EnIuKJUuwZHAP+5uV4d3UMeCveuXKzN7bb2IkX98d4RNz/4M14+ee9Md8WHXk2Gw/f/+B0zA7076Wfz8a1jyIi9sSB7x1ZOxERSx+8szZu/Wg+WveyH/VidotNhc/73V5QfPab2yr7nWMn49KfuxHje2L/905F7Y8pOn88H0ceuCnHeC8cdqPzcf7cfCxlG0vm/iGPujYiYiU6A0F36cvBvXr7n7MSrd9dikvvra8rfxi4vNuMxoWB34FdqBKnX+itF3rq1aivfmlVj1efyg5PvVCNbDgyGePjkfWnpa83CNrYoai1bsT51w/F809PxvJfW3Hjv65F6/P8dQDfoJWVWPm8G0t/bsWH//VOHH1231BAy3SjeX0gjvZ39P68FddGjLkuvXclPnzgGPCr9Xew7dzPx+At6P29nU8v5U4sxc2Psxg8/sTwQtSVzwd3yl2K5S8HfoVdYFPh881rt7LHqHz3aNS2uDh64vHcI1M+ejeOPjsRY/8wG++892HcX4koPXci3v3lwzzPrhQTf5c/Vo2pPRERK9H9a/7cVo3HykdvxtEfHl1XbwqbwKCXZuL53pdnK5/3nnM3UBERsedAHD02+KJRvWyLXpqIoS779ptxdO94xOe3Yv7ZifjOswdi+p8b0TbgAQrTf9TKRHzn2X1x8NU349LALLUho3rR+Eosvb1+zHX0h28O3Dkdj8cnhl4VMfH4cP97gIlvDd+R/DrWv9dUHPi77A5I59PeXV0gYrPhM/7P6bjyUWTTKf7tRtz4xczQTl77j52P+oeX40REXPlrJzv47X1xun8X85VavPrdwbZwJKo/7e1M+6eFePOHB+PQb7Jv/0uTvTsHA9am+85H48/ZIG7/D+pDjy44tDgTB0sRsdKKG79aO741871pchH7v18bfkTCc6fi6pUzAwcAIir/8nzW0+5fi5kne8+5W62X48q9yKZi/dPMul52qr+D9kNpRbsbEVGKqef7+3NPRfV/93Yl7/t2KRuMfdmJdm/wN/VGb2obwKPk3K1Y+jJbOnX0l8Oz7Pb/9Gpc/llExJXeDLfxeP77FweeTnAoav+UTal9kEu3snHmnpfejIvf39qNlL7B96q9snZ86o1aHHo6IuJ+tH7nmaCQt24h6APruVNDOzOmlNLyZ8urG4ytLs5+6WJvc42U0mft3q61vR1nVzfO6C0s7+/gOLCz7d1f9zf4mVpbpP1FJ929fTc1r1VH7nS2ttvtcmou9l+/tqPa0KYcq5sE5XalzW+68cqIz2m10/IXA9es7nY7vDBdKbXb6khvM7KU2r89MuL8wK63/X4x1GOWh3e77fWs1Q3cUrabYvvDrPesbrqRervkfryc0ied4d72g8vZRh0ppeV2v88uZz1scCO3fO9TSqlN1/oNh0bXBmOzwY0j08D4sDe+W+1XAxs9ZmPMbFfw5U862WsfsOHQuh3Gh3a7Xf/+y+1marZvpNqo94rqas8ftdtt+/f9DS1HvfbBx5XaqbW5O58REX96J6ZfmI6T791aXQM6/sR4jD+2EiuftqLxn1fiSkTEB0fjtZ83YunziHhiT5T3TkTnd2/G6duDaypvxa0/34/uE1NRfqYc5WemYvzz+3Hrvddi+p/7WwMtxez/ejdufRoRj5Vi6pmpmPhyJeKD2dj3wmtx6U/3oxulmHqmHOVvj0f3r7fi0v88GPu+4jmkD+392Xj5+Hw07nVjpf85T5di5f6tuPQf7+avBnazY0fjwJ6IiPtx8zf5NUCZxv/7MNvgrHQwDv806zH7XngtLv25GysxHqWn+r2wHUufZq+Z/9HpuHZvJSLGY8/ecpQey/ro6pr5iCg9VY6p8Va8e/xytAc+L947HDP/fiu6KxHje8pRfiri5rl344Y1n8Aj6Nrxl2PmXCOWuisR3+qND7+1Evf/dCne/Y/eReem4+jPG7HU7Y8xp2L83qU4+bOb2fKwB/loPqZfOBrz15eiuxJR+nY5ys+UoxTdWF16f242Tr+/FCtf9vpmaXyD952Pl184GvO/a0X3y6w/l58qxcqnrbh27mgc/O/9DS2BvrFeCgUAAIDCbP7OJwAAAGyS8AkAAEDhhE8AAAAKJ3wCAABQOOETAACAwgmfAAAAFE74BAAAoHDCJwAAAIUTPgEAACic8AkAAEDhhE8AAAAKJ3wCAABQuLGISPmDG/n7v//7/CEAAAD4SpsKnwAAALAVpt0CAABQOOETAACAwgmfAAAAFE74BAAAoHDCJwAAAIUTPgEAACic8AkAAEDhhE8AAAAKJ3wCAABQOOETAACAwgmfAAAAFE74BAAAoHDCJwAAAIUTPgEAACic8AkAAEDhhE8AAAAKJ3wCAABQOOETAACAwgmfAAAAFE74BAAAoHBfL3zO1aPTqUc1fxxgN5qrRyelSK1a/kxmsRkpdaI+lz8BsM3pf8BDSlupaqOTUkopdeqpOuK8Ukrtxsp6YzPV1p2rpnonpU6juu41Sim1E0r/U0o9RK078OBabGahs0/4VEqpgaqlZhoxyFpsbjAoU0qpnVL6n1LqwbXpabfVpyYjuo04OTYWJ69386cBdrnZWLjejdKLM7E2+awa9e+Vo3t9IWaHrgXYSfQ/4KutS6QPW9VGx51PpZRaV7lv/0d96z9XT73FCxvMIsneY1BzMf85Sin1qJX+p5TauDZ95xOAr9L79n//oaiO+tZ/sRnpbCXaF8ZibGwsxsZORiMqcWZ1A7daNNNMTF4/2Ts/FmMXWoMfAPCI0v+AB1uXSB+23PlUSqmNKvvmvtnKf+ufbbyRWrXh63t3ApqL/Z87qT6Xf0+llNoOpf8ppUaXO58Ahci+/S/vzX3rP3coDpS60fhlbvXTuaVoR8TkU9Xez6WonG0OrJsC2C70P2A04ROgIPP32hHRjZvvz68dfHYySlGKytkUKQ3WTJRXL5qNfWMno9Etx0zvfHNx7S0AHnX6HzCK8Anwt3S7Hd3oRuPH/fVOwzVR6Q/U5mN6Ijt28no3ysdSdBrZiiiAbUn/g11P+AT4W+pNKTvwysMPpOYrE7FwJ6I0uXZvAGDb0f9g1xM+Af6m+s/BO5ObSlaNequ32+NcPZpD3/LX4uDeiG7bjo/Adqb/wW4nfAL8jc1XJmLsQivKxwbXPJ2JyQ+noz/pbPLFM0ProSavnxyYkgawPel/sLuN9ba9BQAAgMK48wkAAEDhhE8AAAAKJ3wCAABQOOETAACAwgmfAAAAFE74BAAAoHDCJwAAAIUTPgEAACic8AkAAEDhhE8AAAAKJ3wCAABQOOETAACAwgmfAAAAFE74BAAAoHDCJwAAAIUTPgEAACic8AkAAEDhhE8AAAAKJ3wCAABQOOETAACAwo1FRMof3MiTTz6ZPwQAAABfaVPhEwAAALbCtFsAAAAKJ3wCAABQOOETAACAwgmfAAAAFE74BAAAoHDCJwAAAIUTPgEAACic8AkAAEDhhE8AAAAKJ3wCAABQOOETAACAwgmfAAAAFE74BAAAoHDCJwAAAIUTPgEAACic8AkAAEDhhE8AAAAKJ3wCAABQOOETAACAwgmfAAAAFG4L4bMWzZQirVYn6nP5awB2q2rUO1l/bC7mz/XM1aPzoPMA25L+BzzYJsNnLZppJuLCWIyNZbVwpxSVswIoQF75WDNq+YMAu4D+B4yyyfAZcePCWOw7vvb7bHkhWlGKA69UBy8D2N3utKIV5ZhpGX4Bu4z+B2xgk+FzNmYHgmemFe1uRGmynD8BsIvdiIXr3Yi9h80MAXYZ/Q8YbZPhE4CHNV85HY1uKSpv1eOh5oYsNgfW01tTD2xf+h8wytcPn3OH4kApovXhbP4MwC43H9NvN6JbqsSbjQcPv2qtFOnYZDR+vLam/uT1iMpZG3MA25H+B6z3NcNnLZpnK1G6szC0DhSAnnPTcfp6N0ovvrnxt/hz9Ti8txuNH0/E9Lm1w/OViVi4E1H+3kPeOQB4lOh/QM7Ww+diM1KaiXK3ESfL7noCbGS+cjlaUYrKj0ZvvlF95UCUujfj2sDAq2/2w1ZE6UAc2mjgBvAI0/+AQVsKn9VGJ9KxcnSvn4yxiemYz18AwIDZ2HehFbF3ZuQUsvJkKX8IYIfQ/4A1mw6ftVaKMy+WonVhLCYqYifAQzm+L5tCNuLZd612N3cEYAfR/4CezYXPuXoc3hvRyj3rE4Cvlj0XuRwzbx0YOj5/r73h1LLa8+WIDaakAWwX+h8Qmw2f2bz8RiwIngBb0Jt+VirF0ESz4/ti4U4pKmeHHy1QbXRiZm83Gm9b3gBsd/ofsMnwGRERpUqcGXoOk+cxATy04wvRGDHLbLa89miBfl8982I7FsaGd4AE2Lb0P9j1xiIi5Q8CAADAN2nzdz4BAABgk4RPAAAACid8AgAAUDjhEwAAgMIJnwAAABRO+AQAAKBwwicAAACFEz4BAAAonPAJAABA4YRPAAAACid8AgAAUDjhEwAAgMIJnwAAABRO+AQAAKBwwicAAACFEz4BAAAonPAJAABA4YRPAAAACid8AgAAUDjhEwAAgMKNRUTKH9zIk08+mT8EAAAAX2lT4RMAAAC2wrRbAAAACid8AgAAUDjhEwAAgMIJnwAAABRO+AQAAKBwwicAAACFEz4BAAAonPAJAABA4YRPAAAACid8AgAAUDjhEwAAgMIJnwAAABRO+AQAAKBwwicAAACFEz4BAAAonPAJAABA4YRPAAAACid8AgAAUDjhEwAAgMIJnwAAABRu0+Gz1kqR0mA1o5a/CGA3mqtHJ6VIrQ264mIzUupEfS5/AmAnqEa9k40Pm4v5cz29PrnheWBH22T4rMXBWIixsbFenYxGtxwzAihAxLnpOH29G7H34IieWI3698rRvX46ps/lzwHsLOVjxobAepsMn7Oxrzw78Pt8TL/diG6U46BvsABivnI5WlGOw43q8InFmaiUWnG5Mj98HGCnudOKVpRjZqNZIMCutcnwOcK5pWjnjwHsWrOxcL0bpRdnBr7179/1XIjBr+8AdqYbsXC9G7H3sGUGwJCvHz7npmIyutG+nT8BsDutu/s56q5nf31ovzr1GL5XWovm0Pp6a6SA7WO+cjoa3VJU3sr3tg0sNnN7ilgfDztV2nrVUjOl1GlUR5xTSqndW9VGJ6VOPVWjmuqdXJ9cbKaUUmou9q/Prsmuj9G9dbE5cL1SSj2K1etlrVr2+1w9dfK9rHdssJ/VWiml1En1ubVj1UYn1yeVUjuk1h14cPWaRma4USillOpXFiCbrWZKqZlqq8dzg7N+DQ7I5uqpo78qpbZdre9vWYgc6Gf58PmAfldrDX4pp5TaCbX5abfnpmNidbfb0xFvmRoBsF629rO8N7fWc+5QHCh1o/HL3OrP3vr5yaeqvZ9LUTlrt0hge8uWIZSi8qPR3az6yoEodW/GtRG7gM9+2IooHYhDxpiwY2w+fA6Zj+mJk9mc/g2aCsBuNX+vHRHduPn+wFrPZyejFKWonM0/M3kmyqsXzca+1UdZWe8JbGezse9CK2LvzMg+Vp4s5Q8BO9jXDJ8REfOxdD8i9kw93IJygN3sdju60Y3Gj/szSIZrYnVTovmYnsiOnbzejfKxFJ3841sAtoPj+2Lhzuhnf7ba3dwRYCf7BsJnNab2RMT9pfD0OoCv0JtSe+CVhw+S85WJWLgTUZpcuzcKsJ3MlheyZ3++dWDo+Py99oZTa2vPlyM2mJILbE+bCp/VRmfdlIla60xUSq1YKHt6HcBX6z8H9Eyun1aj3uo9kmCuHs2hu5y1OLg3ottuDRwD2E56029LpRiaaHt8XyzcKUXl7PD+IdVGJ2b2dqPx9rSbG7DDrNuFaMMa2um2v+GtXciUUmpkLTY33hW897iVQcO7Pw7zSCul1KNf63e7HXl+xCNU+o9WWTO4S7hSaqfUWO8HAAAAKMympt0CAADAVgifAAAAFE74BAAAoHDCJwAAAIUTPgEAACic8AkAAEDhhE8AAAAKJ3wCAABQOOETAACAwgmfAAAAFE74BAAAoHDCJwAAAIUTPgEAACic8AkAAEDhhE8AAAAKJ3wCAABQOOETAACAwgmfAAAAFE74BAAAoHDCJwAAAIUbi4iUP7iRJ598Mn8IAAAAvtKmwicAAABshWm3AAAAFE74BAAAoHDCJwAAAIUTPgEAACic8AkAAEDhhE8AAAAKJ3wCAABQOOETAACAwgmfAAAAFE74BAAAoHDCJwAAAIUTPgEAACic8AkAAEDhhE8AAAAKJ3wCAABQOOETAACAwgmfAAAAFE74BAAAoHDCJwAAAIUTPgEAACjc1wyf1ah3UqTUjFr+FMCu1O+LKZqL+XM9c/XoPOg8wHbT62uptcGIcLEZKXWiPpc/AewmXy98Ls5EpZQ/CEBERPmYL+aAXeLcdJy+3o3Ye3BE36tG/Xvl6F4/HdPn8ueA3eRrhM+skQAwwp1WtKIcMxvdBQDYYeYrl6MV5TjcqA6fWJyJSqkVlyvzw8eBXWfr4XNxJirRiIXr3fwZAOJG1h/3HjbNDNglZmPhejdKL84M3P3s3/VciNmha4HdaIvhsxbNY+Vo/dd0tPKnAIiIiPnK6Wh0S1F5qx65+wCjLTYjpWy9aFbWRwHby7q7n6PuevbXh/ark++RtWgO9UJr5GGn2FL4rLVmonxnIfYdz58BYM18TL/diG6pEm/mp6Hl1Fop0rHJaPx4LMbGsjp5PaJy1qAL2E56dz/3H4rqqLuei81IZyvRvtDvdSejEZU4sxpAa9FMMzF5/eRqLxy74FYH7CRpU7XYTCk1U633e7XRGfpdKaV2d1VTvZNSatVWj2V9spPqc71r5uqpk1JqLg7+PnB+oGqtlFKnnqrrPkcppR7VqqVmSqnZGh4zjuqPEbme+IB+qJTa/rW5O59z9egcK0frwj7z9gEeUjYNrRSVH43efKj6yoEodW/GtRG7QM5+2IooHYhDpt8C20Z297O8N3fXc+5QHCh1o/HL3Cjy3FK0I2LyqWrv51JUztotHHaiTYTPatTfqkTJdFuATZqNfRdaEXtnRk6hLU96ZhWws8zfa0dEN26+P7DW89nJKEUpKmeH13OmNBNrz0+YjX1jJ6PRLceM9Z6w4zx8+Ow/03PvzFDDOPNiKSJ6DcIjBQBGO74vFu6MfvZnq23XcGAXuN2ObnSH1rYP1sTqpkTzMT3RX/vejfKxFJ2vWDcPbA8PHz6P71vXJPpNIaIVC2NjMVY2GRdgI7PlhezZn28dGDo+f6+94dTa2vPliA2m5AJsK70ptQdeefggOV+ZiIU7EaVJz5aHneDhwycAX1Nv+m2pFEMTbY/vi4U7paicHX60SrXRiZm93Wi8PR0ezQ5sf/3ngJ7JTaWtRr3V2+12rh7NobuctTi4N6LbtuMt7ATCJ8Df0vGFaIyYZTtbXnu0ytqyhnYsjE3EtLuewA4xX5mIsQutKB8bXPN5JiY/XPuSbfLFM0PrQSevnxyYkgtsZ2O9bW8BAACgMO58AgAAUDjhEwAAgMIJnwAAABRO+AQAAKBwwicAAACFEz4BAAAonPAJAABA4YRPAAAACid8AgAAUDjhEwAAgMIJnwAAABRO+AQAAKBwwicAAACFEz4BAAAonPAJAABA4YRPAAAACid8AgAAUDjhEwAAgMIJnwAAABRO+AQAAKBwwicAAACFEz4BAAAonPAJAABA4YRPAAAACid8AgAAUDjhEwAAgMIJnwAAABRO+AQAAKBwwicAAACF+/9BxRP+1p4OBwAAAABJRU5ErkJggg==)

ACCURACY - OUT OF TOTAL PREDICTIONS HOW MANY ARE RIGHT

Correct Predictions / Total Predictions

**INBALANCED DATA - PRECISION AND RECALL**

PRECISION - Out of all predicted positives, how many actually were positive

Prediction : 100 Customers will Churn but in Reality only 70 did

High Precision = Less False Positive

Precision = TP / (TP + FP)


---


RECALL - Out of all actual positives, how many were identified

100 Customers were to be Churn , Prediction idetified only 70

**Conservative / Aggressive Model**

**BALANCE - F1 SCORE**

![image.png](data:image/png;base64,iVBORw0KGgoAAAANSUhEUgAAA2YAAAC0CAYAAAD/yz9HAAAAAXNSR0IArs4c6QAAAARnQU1BAACxjwv8YQUAAAAJcEhZcwAAEnQAABJ0Ad5mH3gAABTbSURBVHhe7d1PaFx3gifwr5ZecGAbSpCDDbOwWkzjMsnBpi8x0weXyGHc5JAOs5CYGbAlGhZ3Nzhy76Hj6UPWPQu9lgWNuw9BkiFDfBmcQJt4YReVGQL2YRp5wUOVoRurIQ0yZKEKEpAggbeHqpKqSn8sJXaeLX0+8CPS+716r55Bv7zv+/15I0mKAAAAUJp/N7wBAACAb5dgBgAAUDLBDAAAoGSCGQAAQMkEMwAAgJIJZgAAACUTzAAAAEommAEAAJRMMAMAACiZYAYAAFAywQwAAKBkghkAAEDJBDMAAICSCWYAAAAlE8wAAABKJpgBAACUTDADAAAomWAGAABQMsEMAACgZIIZAABAyQQzAACAkglmAM+78wtpFUWK1kKmetu+fy5Xf3Muxwb3BNhbtH/sIYIZwDc2m0ZRpBgqK63lLL5/LmPDuz91b+Xjm1dz7qdXU//f54Yrv6GJfLzcvb5P3hmuTG2+0bn+z+/k0uHhWmDv2U/tX8dss3edrSz8fLMr7P2btLJwfrgOtiaYATxJq6tZ/WI1q6vJgcrBHPu7q1msT33LNyePsvz/VpOsZvnPzeHKrZ25moX7y2kt9z153mAuP/nne0mSA389kQ9e7a+byjuvV5MkSzcv5uKf+uuAPW/Pt3/DKqn9t9ld7A/bE8wAnph26r94IS9894W88MJIfnitmdUklZPncmkgwDxt9Uy+/EJGRl7I0R/Xhyu39oNaai8dTOXAcMWgpZ/9NrceJclYaucn1raP/eZ0apUkX9zN3Fu7OC+wB+yP9m/AV0lerOXC79fbQfgmBDOAp+TW2bncaSfJWKp/M1z7PFvvNTv46rlcPZwkE7n6t50ZHc1/vphfDX4A2Gf2bvu3rv2nZtpJDv7NO/ngteFa2D3BDOBbcKCSJFNZaPXmJZzLjeZKiqJIY76zz9h//SCLn7bW52mstNJ4f3AC+9ibl7OwtL7PymeLuXqob4eu3hyI3rE7juXc+4tZbnXO2zvHnV93v9eZzjDEVGq5XBQpikZm+z/eZ63X7DvHcuofT2XsyrnUDiZp1zN3tvuU+vC5fPCvy2l92T3Xlytp3f8g576/fpxjPx265i9babz/1voOwHNvr7V/a5bncul2O/nOWH70j7OpDdcPO3wuVz95mNZK9/xFkZXWwyxceetbHu7Js6xQFEVRvkmZLRpFURRFq1g437f98NVi8cvu9p+mSKaKhVbn90azVfQ05lOM/XyhaBVFUXzZKh7ebxSN+41ieaVTv/z7ic7xXp0tGt1tvf0eflYUxUp3Y2uhmOqee7a5fuzO9zlVzN7vfbgoVj7tnqO1UjTm3ypmP2kUjU+79Wvf4UbxzoZrXS9jv1ns7P/5YrH4x86Pi78Z6177VLHwWWdba6lzrsbaBX1cTCRFznxcLHe2rO3z8LOiaNWnNpxLUZRntey/9q93/FZ9qsjhS8Wdz4uiKFaKxSvd9m+zf5O+NnHtHL1zFkWx/GH3OpX9XjZsUBRFUXZVNv5P+Nhr7xQ3/jgURNZuTIqiWHlY3Pjpse7nu9u/fFh88FrfcXvB5fM7xaWkeOeT7vE+/bg4d7i337HinU+6B93mxmTsymKxUhRF8eVy8fHPezcPQ2W+cxX9x9m+TBQf95JV0X+dKabqne/08P1Tm+y/Utx5t+98f/ygGFvb51jx1pu1Tc6lKMqzWfZf+zcQzPqP//md4tLhbPpvsvb9P1sopta+f4pT843ud1ssrvZtV/Zt2bBBURRF2VXp/U94EyuNYnbtZmP9xmT5Vt/T0Z92nxZvqVHM5tza09bFK0PnP9/9/DY3JpfvdX4fOO9w2cWNSa+s9ZoNPC1e/65bacynyD/c6dyQFCvF8r/eKC6d6d2oKYry/JT91/4NB7OkttYj16pPFWMbgtk237/v32VjnbLfijlmAE/S6mpWv2hn6d+aufvRr3L65aOZvDm8UzuN23Prv36n+98vmrn1T9dzfUP5MHdzoLtfO61P1z+6U72VxlqP+s77BCz9bDGdBamXsvj2Undr77uupvm/hq+lUz78JMl/P50L/9RM+6sDOfj9H+Wd+cUUny3m6ptmW8BzaZ+1f+vqmfzFh1n6KqmcvJjZ84/S/qK/frvvP52lR52fOnPx2M8EM4Anprdc9Gj+88tHc+JHF3N9q3d5fTW8IcmB1Sy9ezqn/364XMz67cSBvDA68Klk9IXsdIXn0Re/zYU1DmT1Txc3uZ7TuXgtSZby278/mtF/fzyT/+N67v5lNXnxWM79znuB4Pmzz9u/m6fzk4+Wuu82O5UDm11jKhn9j8PbpjJ2MElW0/7LcB37jWAGULaZe1n6qrPC4en3Bl/GeuwfPs6NXyfJh92nqgfyypsf5NTaHqcy+7fHHntjcv1epzfr4KsX88FT75Gazr0/d3469uZspg73VX3/nXz84eUkyVs/fydvHU6Se5n7xemceP3DLCVJ5VC666MBe90eav9u/ZdfdVarPXgsxwZ6v6ZT/7fVJMmxv1sYaBNPzU/kRCXJajN33l/fzv61YXyjoiiKspsyPJ9gq7K+KtnwfmsTwIuiKD572FnFsLti19rKYr25FEVRFJ8vd1YV+7woVj5rdT67zRyLHJ4qFvoW6mgNrEq28fgry42isXynmN1wDcOld+2NwX1f27iCWqO5XKx8WRRFc7ZIb4GQvlXYHvZOvnSjOLXhPIqiPJtl/7V/G+eY9ZX+79l/rZusKrm+KuNK0ZjvXyhJ2a9FjxnAM+DW2R9mYqaepfZq8uJYqi9VU31xNY/+cD2//V13p5nxnP6f9Sy1k/yHg6keGcuBP1/PhV8vpvMsdht/ms74D05n+vZS2qtJ5a+qqb5UTSXtLPWmhs1M5tLNpax+lRw4WE21cuDxx93Kzcn88Ox06n9uZzWVjL1UTfVwJauP7uX6736bJLn3f5t59MWBTt1L1YwdaOfRH67nJ6++kVvDxwP2rD3V/s1Mdt5tNuz/TOboD36S6394lHavTfyrA2n/5V6u/+xEjp7V6pGMdBMaAAAAJdFjBgAAUDLBDAAAoGSCGQAAQMkEMwAAgJIJZgAAACUTzAAAAEommAEAAJRMMAMAACiZYAYAAFAywQwAAKBkghkAAEDJBDMAAICSjSQphjf2+973vje8CQAAgCfoscEMAACAp8tQRgAAgJIJZgAAACUTzAAAAEommAEAAJRMMAMAACiZYAYAAFAywQwAAKBkghkAAEDJBDMAAICSCWYAAAAlE8wAAABKJpgBAACUTDADAAAomWAGAABQMsEMAACgZIIZAABAyQQzAACAkglmAAAAJRPMAAAASiaYAQAAlGwPBrPZNIoirfrUcAUAAMAzadfBbLZZpCiKNOaHa54XU1loFSmKRmaHqxLBDtid8wtpFZ12cavSa0+m6q0Ndc93ewrQu6/aWNbupeYbnW2thWx1dzXb3L4e9oNdBrPZnDjSTrudVF/ZPNY81vmFtIpWFs4PV3xbpjP+bj3tVHNik5uhqfobqbbruVSbHq4C2GhmPKMjIxnplgu320mamevbNjrQngzWjVxrpnrGwyDgOfdgbr1d27TtS1Kp5aK2Dra0u2A2fyLV9mIufdRMjpzYosfpOTAznhsPkurrw09mZjNxspLmR+MRy4BvxdmjmXuQVI6d8qQY2MPaaT5op3LyYokP5+HZtotgNpWF16tp37uV6bN30tyix+l5MflePe1KLRN919DrLZs7278nwNM1ebeZVA6lOlwBsIcsv3cjzVRS+/Fz+2gfnqqdB7Pzp3K80s7izekkk7mzaY9TT2ee1uA44993xiBfqaWSSmpX+scfbzWvqztuudn/B7zx2IP1O7Sh12yT3rLhuSMbxj5vHFe98RoAAEgmc/RaMzkyYW4tbGLHwWzqteOptBdza6bze+cJ7/GcGu6OPr+QVjGRav9Y47frWc6/ZHy083M77dTf3mL88WNM1d9IrvWNYX67nvaRia8ViDq9Zp1r2NBbNt9IcaWW5bVzXUg9tVxeC2dTWWhdTu3R8HUC7M7sK9WkvZzmcAXAXtMdvl09s9UibLB/7TCYTeXUsUpnGGNv09k7aaaS46/1B6KpLPyylsqDuYxUJ9c3z4zn6C4D2Fama6M52j/UsNvzVTn0NQYBzYznxoNKaj9uDPWWdYZt5sFc37m6i4asDX+s5lAlad59OtcJ7A9T9VYmjrRTf9fcVuA5dmRicDTTNgu9TVbn0kw1E19nxBPsYTsLZvMTqa0NY+zpDGccmLB+/lSOD4eVp2JwCOHEkSQHx7YYVrm9yffqaR+pDvaWdYdt1t8buo6ZpSwnOfSfppI0s9xOqme2bngANqpmou/m5fLJ5cyNjGa8OxoB4Lm0YVXG7dq1yczdbidH3nAPBX12FMxmX6kmffPCBgJR/wIaLx9KJe0s3x86wBPUeY/a5Ry/d2Htj3/uwfBeu9ANWwO9gS8fGpgHt14m+ibnT2d8dCRzD4bnywFsZ2i5/JGjedqPsgCeNdO1S6m3K6n9cnj+PuxfOwhmszlxJGnfXg9C62UuzfS90+z+ctrDH9+RTu/TY803OkN+3t793LRdub88MA9uuPSfe7La3X6tmcrJy19vIRIAgH1lfYqId5tBx+OD2fyJVDM8jLGn1xXdfafZzFKWN8w727kN88S6QyO31wmOT9TXuY6zRzsvlv2aQyoBAPaVmfFcut15t9kbB4crYf95TDDrLoLRtxrjsOmbi2mvvdOsswxq5eTlwWF95xfS6P2+aeiZzq177aHlU7sLifTt1enJGvzsbLN/eOGT0gmclZOXh5ZzncpCc315/cZA71hngZQ8WjKBHwBgB3pDGiuPfRAPe9/2wazbYzUw/2rYzK0stvveB3b2aEbericnL6/PzbpyPMtrPW7road/XtZ0bbS7fGpvPtfF5N0LqfcPcZwZz2hvyGD32CfufsM5ZluYro1m5Fqz7/t05rYdutu3ctrACkTdpfP7V6MEAGAb0xn/yMtCIElGkhTDGwEAAPj2bN9jBgAAwFMnmAEAAJRMMAMAACiZYAYAAFAywQwAAKBkghkAAEDJBDMAAICSCWYAAAAlE8wAAABKJpgBAACUTDADAAAomWAGAABQMsEMAACgZIIZAABAyQQzAACAkglmAAAAJRPMAAAASiaYAQAAlEwwAwAAKNlIkmJ4Y7/vfve7w5sAAAB4gh4bzAAAAHi6DGUEAAAomWAGAABQMsEMAACgZIIZAABAyQQzAACAkglmAAAAJRPMAAAASiaYAQAAlEwwAwAAKJlgBgAAUDLBDAAAoGSCGQAAQMkEMwAAgJIJZgAAACUTzAAAAEommAEAAJRMMAMAACiZYAYAAFAywQwAAKBkghkAAEDJ9mAwm02jKNKqTw1XAAAAPJN2Hcxmm0WKokhjfrjm+TJVb6UoihTN2eGqrqkstLarB+jpthfFxrL2kGi+saFuoB7geXN+Ia1N2rXN2jj3XfB4uwxmszlxpJ12O6m+8jX/cM4vpFW0snB+uKIkRyae+5AJPCMezGVkZGSgjNam+3Zop/52X/3b9eTkZTciwPNpZjyjfe3dhdvtJM3MbdkGuu+C7ewumM2fSLW9mEsfNZMjJ/L830o003yQVM809sC1AM+dmfFcut3eI+0pwOO474Lt7CKYTWXh9Wra925l+uydNFPNiT3wxOPOe/W0U80bhhMBJZi+uZh2DmXsWRlFAPAUue+Cre08mJ0/leOVdhZvTieZzJ0HSfX1hWz+Z9VZgGNwjPHvO2OHr9RSSSW1K/1jj7dasGOz8cYbj/2NhgF1n1hXTl7c4fDKjeff+L0BANhg1/ddsH/sOJhNvXY8lfZibs10fp+820wqx3Nq+I/q/EJaxUSq/XMt3q5nOf+S8dHOz+2+eRYbxh4/xlT9jeTa4ByN9pGJbxSOpmuXUm9XUvvlVkGza76RopjIodsXzBEBnoip146nkuUsddtWgL1ux/ddsM/sMJhN5dSxSmcYY2/T2TtpppLjr/X/SU1l4Ze1VB7MZaQ6ub55ZjxHdxnAtjJdG83Rs30bZsZz40FSOVTt27hb0xl/t552pZaLWwa87lDO2xcGw+TMeEavNZMjb3jyA/vdkYmhFckes9DRfCOXT1bSvHY0fS0mwB63k/su2H92FszmJ1JbG8bY0xnOWDl2av1px/lTOV5Jmnef9i3G4NLUE0eSHBz7Zk9degHv5MTmE1IHhnIO2TSkAvvOhlUZRzM+0BO2Poy7KIoUZw6l/vbI4MMmgP3gcfddsA/tKJjNvlLdeEPRC0SVWiZ6i4C8fCiVtLN8f+gAT1DnPWqXc/ze+nDCuQfDe309k9W5NFPNxGbDEl8+lMrwNoBdGVouf0NwA9g/tr3vgn1oB8FsNieOJO3+eVVrZS7N9L3T7P5y2sMf35FmlnfywflGJo50bmx2OzdtZyZz9Fpz83dsfO1rAwBgo23uu2Afenwwmz+RarYYwpfJzPW/g2dmKcvfYEjfhnli3aGR2+sExyfm7NHMPUiqZy7meP/27a5t238jAAA2tdV9F+xDjwlmnQUv0rca47DOO3h67zTrPPmonLw8uEri+YU0er9vGnCmc+tee+iJSXchkb69Or1Wg5+dbU7kmyz7sZlO13ollYGTb31trTPVtG9fMiQJAGCXNr/vgv1n+2DW7bEaWI1x2MytLLb73ml29uj6EvK9+WhXjmd5rTep08tW6db3Qs50bbT7xKQ3h+1i8u6F1PvHD3ZXQOx9tiiKnLj75OaYrev2BA7b9NpqWb72tIZWAgDsdVvcd8E+M5KkGN4IAADAt2f7HjMAAACeOsEMAACgZIIZAABAyQQzAACAkglmAAAAJRPMAAAASiaYAQAAlEwwAwAAKJlgBgAAUDLBDAAAoGSCGQAAQMkEMwAAgJIJZgAAACUTzAAAAEommAEAAJRMMAMAACiZYAYAAFAywQwAAKBkghkAAEDJBDMAAICSCWYAAAAlE8wAAABKJpgBAACUTDADAAAomWAGAABQMsEMAACgZIIZAABAyQQzAACAkglmAAAAJRPMAAAASiaYAQAAlEwwAwAAKJlgBgAAUDLBDAAAoGSCGQAAQMkEMwAAgJIJZgAAACUTzAAAAEr2/wHpkB+LrVyHkQAAAABJRU5ErkJggg==)

In [ ]:
"""

#ROC Curve, Dashboard, Feature Importance & Persistence

fpr_lr, tpr_lr, _ = roc_curve(y_test, prob_lr)
fpr_dt, tpr_dt, _ = roc_curve(y_test, prob_dt)
fpr_rf, tpr_rf, _ = roc_curve(y_test, prob_rf)

plt.figure(figsize=(8,6))

plt.plot(
    fpr_lr,
    tpr_lr,
    label=f'Logistic (AUC = {roc_auc_score(y_test, prob_lr):.3f})'
)

plt.plot(
    fpr_dt,
    tpr_dt,
    label=f'Decision Tree (AUC = {roc_auc_score(y_test, prob_dt):.3f})'
)

plt.plot(
    fpr_rf,
    tpr_rf,
    label=f'Random Forest (AUC = {roc_auc_score(y_test, prob_rf):.3f})'
)

plt.plot([0,1],[0,1],'k--')

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve Comparison")
plt.legend()

plt.show()

from sklearn.metrics import precision_recall_curve, average_precision_score
import matplotlib.pyplot as plt

# Precision-Recall values
precision_lr, recall_lr, _ = precision_recall_curve(y_test, prob_lr)
precision_dt, recall_dt, _ = precision_recall_curve(y_test, prob_dt)
precision_rf, recall_rf, _ = precision_recall_curve(y_test, prob_rf)

plt.figure(figsize=(8,6))

plt.plot(
    recall_lr,
    precision_lr,
    label=f'Logistic (AP = {average_precision_score(y_test, prob_lr):.3f})'
)

plt.plot(
    recall_dt,
    precision_dt,
    label=f'Decision Tree (AP = {average_precision_score(y_test, prob_dt):.3f})'
)

plt.plot(
    recall_rf,
    precision_rf,
    label=f'Random Forest (AP = {average_precision_score(y_test, prob_rf):.3f})'
)

plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve")

plt.legend()

plt.show()

In [ ]:
"""How well the model separates the Positive class from the Negative class across all possible thresholds.

In [ ]:
"""

comparison = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Decision Tree",
        "Random Forest"
    ],
    "Accuracy": [
        accuracy_score(y_test, pred_lr),
        accuracy_score(y_test, pred_dt),
        accuracy_score(y_test, pred_rf)
    ],
    "Precision": [
        precision_score(y_test, pred_lr),
        precision_score(y_test, pred_dt),
        precision_score(y_test, pred_rf)
    ],
    "Recall": [
        recall_score(y_test, pred_lr),
        recall_score(y_test, pred_dt),
        recall_score(y_test, pred_rf)
    ],
    "F1 Score": [
        f1_score(y_test, pred_lr),
        f1_score(y_test, pred_dt),
        f1_score(y_test, pred_rf)
    ],
    "ROC-AUC": [
        roc_auc_score(y_test, prob_lr),
        roc_auc_score(y_test, prob_dt),
        roc_auc_score(y_test, prob_rf)
    ]
})

comparison.round(3)

# Feature Importance - Random Forest
feature_importance = pd.Series(
    rf.feature_importances_,
    index=X_train.columns
).sort_values(ascending=False)
display(
    pd.DataFrame(feature_importance, columns=["Importance"]).head(10)
)
plt.figure(figsize=(8,5))
feature_importance.head(10).plot(
    kind="barh",
    color="forestgreen"
)
plt.gca().invert_yaxis()
plt.title("Top 10 Important Features - Random Forest")
plt.xlabel("Importance Score")
plt.show()

In [ ]:
"""# Model Selection

We trained and evaluated three classification models:

- Logistic Regression
- Decision Tree
- Random Forest

Each model was assessed using multiple evaluation metrics including:

- Accuracy
- Precision
- Recall
- F1 Score
- ROC-AUC
- Precision-Recall Curve

Rather than selecting a model solely based on accuracy, the final model should align with the business objective.

Since our business problem focuses on identifying customers likely to churn, minimizing False Negatives is more important than simply maximizing overall accuracy.

Therefore, Recall becomes the primary evaluation metric.

The next section compares all trained models and selects the most suitable model for deployment.

In [ ]:
"""

model_selection = comparison.copy()

model_selection = model_selection.sort_values(
    by="Recall",
    ascending=False
)

display(model_selection)

best_model_name = model_selection.iloc[0]["Model"]

print(f"Selected Model for Deployment : {best_model_name}")

In [ ]:
"""# Hyperparameter Tuning

Until now, we have trained machine learning models using manually selected values such as:

- n_estimators = 100
- max_depth = 10
- k = 5

But an important question arises:

**How do we know these values are actually the best?**

The answer is:

We don't.

These values are called **Hyperparameters**.

Instead of manually guessing them, we use Hyperparameter Tuning techniques to automatically search for the best combination.

Hyperparameter tuning helps us improve:

- Accuracy
- Precision
- Recall
- Generalization
- Model Stability

In this section we will learn:

• Parameters vs Hyperparameters

• Grid Search

• Random Search

• Choosing the Best Model

# Parameters vs Hyperparameters

Machine Learning models contain two different types of values.

## Parameters

These are values that the model learns automatically from the training data.

Examples:

- Weights
- Bias
- Decision boundaries

The programmer never manually sets these values.

---------------------------------------

## Hyperparameters

These are values decided before training starts.

Examples:

- Number of Trees
- Maximum Depth
- Learning Rate
- Number of Neighbors
- Kernel Type

The programmer chooses these values.

Hyperparameter tuning automates this selection process.

In [ ]:
"""

from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# Baseline Decision Tree
baseline_dt = DecisionTreeClassifier(random_state=42)

baseline_dt.fit(X_train, y_train)

baseline_pred = baseline_dt.predict(X_test)

baseline_accuracy = accuracy_score(y_test, baseline_pred)

print(f"Baseline Accuracy : {baseline_accuracy:.4f}")

param_grid = {
    "criterion": ["gini", "entropy"],
    "max_depth": [3, 5, 10, 15, None],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": [None, "sqrt", "log2"]
}

param_grid

In [ ]:
"""# Grid Search

Grid Search is an exhaustive search technique.

It trains the model using every possible combination of hyperparameters.

The model with the best Cross Validation score is selected.

Although Grid Search is computationally expensive,

it guarantees that every specified combination is evaluated.

In [ ]:
"""

from sklearn.model_selection import GridSearchCV

grid_search = GridSearchCV(
    estimator=DecisionTreeClassifier(random_state=42),
    param_grid=param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

print("Best Parameters")
print(grid_search.best_params_)

print("\nBest Cross Validation Score")
print(round(grid_search.best_score_,4))

#Tuned Decision Tree
best_dt = grid_search.best_estimator_
pred_tuned = best_dt.predict(X_test)
tuned_accuracy = accuracy_score(y_test, pred_tuned)

print("Test Accuracy :", round(tuned_accuracy,4))

In [ ]:
"""# Comparing Baseline vs Tuned Model

Hyperparameter tuning should improve the model's ability to generalize.

Rather than relying on manually selected values,

we now compare:

- Baseline Model
- Tuned Model

to determine whether tuning provided measurable improvement.

In [ ]:
"""

tuning_comparison = pd.DataFrame({

    "Model":[
        "Baseline Decision Tree",
        "Tuned Decision Tree"
    ],

    "Accuracy":[
        baseline_accuracy,
        tuned_accuracy
    ]
})

tuning_comparison

plt.figure(figsize=(6,4))
sns.barplot(
    data=tuning_comparison,
    x="Model",
    y="Accuracy"
)
plt.title("Baseline vs Tuned Decision Tree")
plt.ylim(0,1)
plt.show()

In [ ]:
"""# Limitations of Grid Search

Grid Search evaluates every possible combination.

While this ensures a thorough search,

it also makes Grid Search computationally expensive for large search spaces.

For models with many hyperparameters,

training time may become very high.

To solve this problem,

we use Randomized Search.

In [ ]:
"""

from sklearn.model_selection import RandomizedSearchCV

random_search = RandomizedSearchCV(
    estimator=DecisionTreeClassifier(random_state=42),
    param_distributions=param_grid,
    n_iter=20,
    cv=5,
    scoring="accuracy",
    random_state=42,
    n_jobs=-1
)

random_search.fit(X_train, y_train)

print("Best Parameters")
print(random_search.best_params_)

print("\nBest Cross Validation Score")
print(round(random_search.best_score_,4))

In [ ]:
"""# Grid Search vs Random Search

Grid Search

• Evaluates every possible combination

• Higher computational cost

• Better for small search spaces

---------------------------------------

Random Search

• Evaluates only randomly selected combinations

• Much faster

• Preferred for large search spaces

In modern machine learning projects,

Random Search is often used as a practical alternative to Grid Search.

In [ ]:
"""

best_random_dt = random_search.best_estimator_
pred_random = best_random_dt.predict(X_test)
random_accuracy = accuracy_score(y_test, pred_random)

print("Random Search Test Accuracy :", round(random_accuracy,4))

search_comparison = pd.DataFrame({

    "Model":[
        "Baseline DT",
        "Grid Search DT",
        "Random Search DT"
    ],

    "Accuracy":[
        baseline_accuracy,
        tuned_accuracy,
        random_accuracy
    ]
})

search_comparison

plt.figure(figsize=(7,5))

sns.barplot(
    data=search_comparison,
    x="Model",
    y="Accuracy",
    palette="Set2"
)

plt.title("Decision Tree Hyperparameter Tuning Comparison")
plt.ylim(0,1)
plt.show()

In [ ]:
"""# Visualizing the Final Decision Tree

After selecting and tuning the Decision Tree model, we can visualize its learned decision-making process.

Each internal node represents a decision based on a feature.

The colored leaf nodes represent the final prediction.

Tree visualization helps us understand how the model reaches a prediction and improves model interpretability.

In [ ]:
"""

from sklearn.tree import plot_tree

plt.figure(figsize=(22,10))

plot_tree(
    best_dt,
    feature_names=X_train.columns,
    class_names=["No Churn","Churn"],
    filled=True,
    rounded=True,
    max_depth=3,
    fontsize=9
)

plt.title("Decision Tree Visualization (First 3 Levels)")
plt.show()

In [ ]:
"""# Machine Learning Pipeline

Until now, every preprocessing step was performed manually.

- Missing Value Handling
- Encoding
- Feature Scaling
- Model Training

In production, these steps are combined into a single Pipeline.

A Pipeline ensures that every new input passes through the exact same preprocessing steps before reaching the trained model.

Advantages:

- Cleaner Code
- Less Human Error
- Easier Deployment
- Consistent Predictions

In [ ]:
"""

from sklearn.pipeline import Pipeline

pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", DecisionTreeClassifier(**grid_search.best_params_, random_state=42))
])

pipeline.fit(X_train, y_train)

In [ ]:
"""# Model Serialization

Training a Machine Learning model can take significant time.

Instead of training the model every time the application starts, we save the trained model to disk.

This process is called **Model Serialization**.

The saved model can later be loaded and used directly for making predictions without retraining.

In Scikit-Learn, the Joblib library is commonly used for model serialization.

In [ ]:
"""

joblib.dump(pipeline, "best_model.pkl")
print("Pipeline saved successfully.")

In [ ]:
"""# Loading the Saved Model

Once the model has been saved, it can be loaded back into memory whenever predictions are required.

This avoids retraining and significantly reduces deployment time.

In [ ]:
"""

loaded_pipeline = joblib.load("best_model.pkl")
print("Pipeline loaded successfully.")

In [ ]:
"""# Model Inference

Training is the process where the model learns from historical data.

Inference is the process of using the trained model to make predictions on new, unseen data.

During deployment, only inference takes place.

In [ ]:
"""

sample_customer = X_test.iloc[[0]]

prediction = loaded_pipeline.predict(sample_customer)
probability = loaded_pipeline.predict_proba(sample_customer)

print("Prediction :", prediction[0])
print("Prediction Probabilities :", probability)

sample_customer = X_test.iloc[[0]]

display(sample_customer)

prediction = loaded_pipeline.predict(sample_customer)

if prediction[0] == 1:
    print("⚠️ Customer is likely to Churn")
else:
    print("✅ Customer is likely to Stay")

In [ ]:
"""# AI Assisted Deployment

Modern Machine Learning Engineers use AI coding assistants such as:

- ChatGPT
- Claude
- GitHub Copilot

These tools accelerate application development.

Our next step is to integrate the trained pipeline into a professional Streamlit application.

# Prompt for AI

You are a Senior Machine Learning Engineer.

I have a trained Decision Tree Pipeline saved as **best_model.pkl**.

Create a professional Streamlit application with:

- Modern UI
- Sidebar
- Customer Input Form
- Predict Button
- Prediction Probability
- Error Handling
- Download Prediction
- Professional Styling

# Production Project Structure

Customer_Churn_Project/

│

├── app.py

├── best_model.pkl

├── requirements.txt

├── README.md

├── customer_data.csv

└── notebook.ipynb

In [ ]:
"""

import joblib

joblib.dump(model, "churn_model.pkl")